# Molecular Descriptors and Fingerprints

This notebook accompanies the QSAR tutorial chapter: **Molecular Descriptors and Fingerprints**.

## Calculate simple RDKit descriptors

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

smiles = "CC(C)Cc1ccc(cc1)C(C)C(=O)O"  # ibuprofen
mol = Chem.MolFromSmiles(smiles)

features = {
    "MolWt": Descriptors.MolWt(mol),
    "MolLogP": Descriptors.MolLogP(mol),
    "TPSA": Descriptors.TPSA(mol),
    "NumHDonors": Descriptors.NumHDonors(mol),
    "NumHAcceptors": Descriptors.NumHAcceptors(mol),
    "NumRotatableBonds": Descriptors.NumRotatableBonds(mol),
}

features

## Calculate all default RDKit descriptors for one molecule

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors
import pandas as pd
import numpy as np

smiles = "CC(C)Cc1ccc(cc1)C(C)C(=O)O"  # ibuprofen
mol = Chem.MolFromSmiles(smiles)

if mol is None:
    raise ValueError("Invalid SMILES string.")

all_descriptors = Descriptors.CalcMolDescriptors(mol, missingVal=np.nan)

descriptor_table = pd.DataFrame(
    all_descriptors.items(),
    columns=["Descriptor", "Value"]
)

print("Number of descriptors calculated:", len(descriptor_table))
display(descriptor_table.head(15))

## Morgan fingerprints

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

smiles = "CCO"
mol = Chem.MolFromSmiles(smiles)

fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)

arr = np.zeros((1024,), dtype=int)
Chem.DataStructs.ConvertToNumpyArray(fp, arr)

print("Fingerprint length:", len(arr))
print("Number of active bits:", arr.sum())
print("First 30 bits:", arr[:30])

## Calculate RDKit descriptors for the full example dataset

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import Descriptors

def find_example_dataset():
    candidates = [
        Path("../data/example_qsar_dataset.csv"),
        Path("data/example_qsar_dataset.csv"),
        Path("../../data/example_qsar_dataset.csv"),
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("Could not find example_qsar_dataset.csv.")

def mol_from_smiles(smiles):
    try:
        return Chem.MolFromSmiles(smiles)
    except Exception:
        return None

def calculate_all_rdkit_descriptors(mol):
    return Descriptors.CalcMolDescriptors(mol, missingVal=np.nan)

# Load data
df = pd.read_csv(find_example_dataset())

# Convert SMILES to RDKit molecules and remove invalid molecules
df["Mol"] = df["SMILES"].apply(mol_from_smiles)
df_clean = df.dropna(subset=["Mol"]).copy()

# Calculate all default RDKit descriptors
descriptor_rows = [
    calculate_all_rdkit_descriptors(mol)
    for mol in df_clean["Mol"]
]

X = pd.DataFrame(descriptor_rows, index=df_clean.index)

# Convert to numeric and remove descriptor columns containing missing values
X = X.apply(pd.to_numeric, errors="coerce")
X = X.dropna(axis=1)

# Target values
y = df_clean["Activity"].astype(float)

print("Number of valid molecules:", len(df_clean))
print("Number of RDKit descriptors:", X.shape[1])

X.head()

In [ ]:
# Save the descriptor matrix for later chapters if desired
X.to_csv("rdkit_descriptors_for_example_dataset.csv", index=False)
print("Saved descriptor matrix to rdkit_descriptors_for_example_dataset.csv")